# ⚡ Tutorial 2: Parameter-Efficient Fine-Tuning (PEFT LoRA) & 6-Gate Assurance on Kilat

This tutorial demonstrates how to adapt a **Kilat** model using Hugging Face **PEFT (Low-Rank Adaptation / LoRA)** while preserving Kilat's mathematical invariants:
1. **Recurrent Horizon Protection**: Kilat's Global Exponential Decay parameters (`log_lambda`) govern recurrent horizon stability and must **never** be wrapped by low-rank linear adapters.
2. **Canonical Target Modules**: `get_kilat_lora_config()` automatically targets SwiGLU FFN projections (`gate`, `up`, `down`), Latent MLA projections (`q_proj`, `q_a_proj`, `q_b_proj`, `kv_a_proj`, `kv_b_proj`, `o_proj_recall`), and Global Decay linear projections (`v_proj_global`, `o_proj_global`).
3. **6-Gate Assurance on `PeftModel`**: Verify that the LoRA-wrapped model passes all 7 verification gates (`G0`–`G6`), including 2-step LoRA gradient connectivity.
4. **Zero-Overhead Deployment**: Merge LoRA weights back into the base `KilatForCausalLM` via `merge_and_unload()`.

In [ ]:
# Uncomment to install in Google Colab:
# !pip install -q git+https://github.com/Airukua/Kilat.git peft transformers safetensors

## 1. Wrap `KilatForCausalLM` with `get_kilat_lora_config()`

Let's instantiate a `KilatForCausalLM` model and apply `get_kilat_lora_config(r=8, lora_alpha=16)`.

In [ ]:
import tempfile
from pathlib import Path
import torch
from peft import PeftModel, get_peft_model

from kilat.integrations.huggingface import (
    DEFAULT_KILAT_LORA_TARGET_MODULES,
    get_kilat_lora_config,
    register_kilat,
)
from kilat.integrations.huggingface.configuration_kilat import KilatConfig
from kilat.integrations.huggingface.modeling_kilat import KilatForCausalLM

torch.manual_seed(42)
register_kilat()

hf_config = KilatConfig(
    vocab_size=128,
    n_embd=64,
    n_layer=2,
    n_head=4,
    recall_ratio=0.5,
    recall_pattern="macro_synergy",
    ffn_mode="swiglu",
    use_rope=True,
)

base_model = KilatForCausalLM(hf_config)
base_snapshot = {k: v.detach().clone() for k, v in base_model.state_dict().items()}
lora_config = get_kilat_lora_config(r=8, lora_alpha=16, lora_dropout=0.0)
peft_model = get_peft_model(base_model, lora_config)

print("Default LoRA Target Modules:", DEFAULT_KILAT_LORA_TARGET_MODULES)
peft_model.print_trainable_parameters()

## 2. Verify the `PeftModel` with Kilat's 6-Gate Assurance Pipeline (`G0`–`G6`)

Kilat's assurance pipeline natively unwraps `PeftModel` to verify:
- No LoRA adapters are attached to forbidden modules (`router`, `rms1`, `rms2`, `ln_f`, `wte`, `log_lambda`).
- Causal future-blindness Jacobian on frozen input embeddings.
- 2-step LoRA gradient connectivity (accounting for `lora_B = 0` at initialization).

In [ ]:
from kilat.assurance.runner import run_verification_pipeline
from kilat.configs.model_config import KilatConfig as KilatNativeConfig

native_cfg = KilatNativeConfig(**hf_config.to_dict())
with tempfile.TemporaryDirectory() as tmp_dir:
    assurance_report = run_verification_pipeline(
        config=native_cfg,
        model=peft_model,
        output_dir=tmp_dir,
    )
    print("PEFT 6-Gate Assurance Status:", assurance_report.overall_status)
    for g in assurance_report.gates:
        print(f"  [{g.status}] {g.gate_id}: {g.metrics}")

## 3. Fine-Tune, Save Adapter, and Merge Weights (`merge_and_unload`)

We now run a few optimization steps on the LoRA adapter, save only the lightweight adapter weights to disk, reload them onto a fresh base model, and merge them via `merge_and_unload()` for zero-latency inference.

In [ ]:
optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=5e-3)

batch_ids = torch.randint(0, hf_config.vocab_size, (4, 16))
peft_model.train()
for step in range(10):
    optimizer.zero_grad(set_to_none=True)
    out = peft_model(input_ids=batch_ids, labels=batch_ids)
    out.loss.backward()
    optimizer.step()
    if (step + 1) % 5 == 0:
        print(f"Step {step + 1:02d} | Loss: {out.loss.item():.4f}")

peft_model.eval()
with tempfile.TemporaryDirectory() as adapter_dir:
    peft_model.save_pretrained(adapter_dir)
    print("Saved adapter files:", [p.name for p in Path(adapter_dir).iterdir()])

    # Reload onto a fresh base model and merge into standard linear layers
    fresh_base = KilatForCausalLM(hf_config)
    fresh_base.load_state_dict(base_snapshot, strict=True)
    reloaded = PeftModel.from_pretrained(fresh_base, adapter_dir)
    merged_model = reloaded.merge_and_unload()
    merged_model.eval()

    with torch.no_grad():
        logits_peft = peft_model(input_ids=batch_ids).logits
        logits_merged = merged_model(input_ids=batch_ids).logits
        max_diff = float((logits_peft - logits_merged).abs().max().item())
    print(f"Merged model type: {type(merged_model).__name__} | Max logit diff: {max_diff:.3e}")